# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_17 — Transformer vs LSTM for Returns: Warning Case

### Research question

When comparing LSTMs and Transformers for financial return forecasting, how can we design a benchmark that highlights overfitting, random-split optimism, and the danger of overclaiming predictability?

This notebook follows naturally from:

```text
03_06_arima_vs_lstm_benchmark
03_07_lstm_forecaster
03_14_information_coefficient_and_feature_decay
03_15_purged_feature_selection_for_time_series
03_16_deep_hedging_intro_pytorch
```

The aim is deliberately cautious.

It does **not** ask:

> Which deep model predicts markets best?

It asks:

> How can a Transformer appear impressive in a flawed benchmark, and how does that performance change under chronological validation, naïve baselines, regime stress, and cost-aware diagnostics?

It covers:

1. noisy financial return simulation;
2. sequence-window dataset construction;
3. train-only scaling;
4. chronological train/validation/test split;
5. random split warning case;
6. naïve baselines;
7. LSTM forecaster in PyTorch;
8. Transformer encoder forecaster in PyTorch;
9. parameter-count comparison;
10. training and validation curves;
11. out-of-sample metrics;
12. directional accuracy and IC;
13. regime-dependent errors;
14. placebo target check;
15. toy strategy diagnostics;
16. cost sensitivity;
17. model-card warning labels;
18. saved outputs and manifest.

The key idea:

> More architecture does not create signal. In noisy return forecasting, validation design matters more than whether the model is an LSTM or a Transformer.

## 1. Why this notebook is a warning case

Transformers are powerful.

They are excellent in domains where long-context structure is abundant:

- language;
- vision;
- some sequence modelling tasks;
- large-scale representation learning.

Financial return forecasting is different:

- signal-to-noise is low;
- relationships are unstable;
- labels are noisy;
- regimes shift;
- costs matter;
- random splits leak temporal structure;
- large models overfit easily.

A Transformer may outperform an LSTM in a flawed setup simply because:

1. random splitting makes train and test too similar;
2. scaling uses future data;
3. overlapping labels leak;
4. the model has more parameters;
5. the evaluation ignores costs;
6. the test period is accidentally easy.

This notebook is designed to make those risks visible.

## 2. Forecasting target

We forecast next-period return:

$$
r_{t+1}
$$

from a trailing feature sequence:

$$
X_t = [x_{t-L+1},\dots,x_t]
$$

where $L$ is sequence length.

The target is noisy by construction.

A good model should be judged against:

- zero forecast;
- train mean forecast;
- last return;
- rolling mean;
- small linear baseline.

If a deep model cannot beat these baselines out of sample, it is not useful.

## 3. LSTM versus Transformer

### LSTM

An LSTM processes a sequence recursively:

$$
h_t = f(x_t,h_{t-1})
$$

It has an inductive bias toward sequential memory.

### Transformer encoder

A Transformer uses self-attention:

$$
Attention(Q,K,V) = softmax\Big(\frac{QK^\top}{\sqrt{d}}\Big)V
$$

It can relate all time steps in the window to each other.

But attention can also overfit when:

- data is small;
- signal is weak;
- validation is flawed;
- model capacity is excessive.

This notebook compares them as forecasting tools, not as magic alpha machines.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

TORCH_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class TransformerLSTMWarningConfig:
    n_obs: int = 3_200
    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    sequence_length: int = 48
    annualisation: int = 252
    batch_size: int = 128
    epochs: int = 25
    learning_rate: float = 0.001
    hidden_dim: int = 32
    transformer_dim: int = 32
    transformer_heads: int = 4
    transformer_layers: int = 2
    dropout: float = 0.15
    cost_per_turnover: float = 0.0005
    seed: int = 42


config = TransformerLSTMWarningConfig()
config

## 5. Synthetic market with weak and unstable predictability

We simulate a series with:

1. weak latent trend;
2. short-term reversal;
3. volatility clustering;
4. regime changes;
5. heavy-tailed shocks;
6. changing signal strength through time.

This is intentionally realistic in spirit:

> There is some signal, but it is weak, noisy, and unstable.

In [ ]:
def simulate_warning_case_market(config: TransformerLSTMWarningConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs

    returns = np.zeros(n)
    latent_trend = np.zeros(n)
    latent_reversal = np.zeros(n)
    latent_vol = np.zeros(n)
    regime = np.zeros(n, dtype=int)

    latent_vol[0] = 0.010

    transition = np.array([
        [0.985, 0.015],
        [0.035, 0.965]
    ])

    for t in range(3, n):
        regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        latent_trend[t] = 0.975 * latent_trend[t - 1] + 0.08 * rng.standard_normal()
        latent_reversal[t] = -0.45 * returns[t - 1] + 0.10 * rng.standard_normal()

        base_vol = 0.008 if regime[t] == 0 else 0.022
        latent_vol[t] = (
            0.93 * latent_vol[t - 1]
            + 0.07 * base_vol
            + 0.08 * abs(returns[t - 1])
        )
        latent_vol[t] = np.clip(latent_vol[t], 0.003, 0.070)

        # Signal is weak and changes sign slightly late in the sample.
        signal_strength = 0.00035 if t < int(0.72 * n) else 0.00005
        reversal_strength = 0.00022 if regime[t] == 0 else -0.00005

        predictable_mean = (
            0.00005
            + signal_strength * latent_trend[t - 1]
            + reversal_strength * latent_reversal[t - 1]
            - 0.00025 * max(latent_vol[t] * np.sqrt(252) - 0.30, 0)
        )

        seasonal_component = 0.00010 * np.sin(2 * np.pi * t / 63)

        shock = latent_vol[t] * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)

        returns[t] = predictable_mean + seasonal_component + shock

        # Rare negative jumps.
        if rng.uniform() < 0.003:
            returns[t] -= rng.uniform(0.015, 0.050)

    dates = pd.bdate_range("2015-01-01", periods=n)
    price = 100 * np.exp(np.cumsum(returns))

    df = pd.DataFrame({
        "date": dates,
        "return": returns,
        "price": price,
        "latent_trend": latent_trend,
        "latent_reversal": latent_reversal,
        "latent_vol": latent_vol,
        "regime": regime
    })

    # Backward-looking features.
    df["abs_return"] = df["return"].abs()
    df["squared_return"] = df["return"] ** 2
    df["momentum_5"] = np.log(df["price"] / df["price"].shift(5))
    df["momentum_21"] = np.log(df["price"] / df["price"].shift(21))
    df["momentum_63"] = np.log(df["price"] / df["price"].shift(63))
    df["rolling_mean_5"] = df["return"].rolling(5).mean()
    df["rolling_mean_21"] = df["return"].rolling(21).mean()
    df["rolling_vol_21"] = df["return"].rolling(21).std()
    df["rolling_vol_63"] = df["return"].rolling(63).std()
    df["vol_ratio_21_63"] = df["rolling_vol_21"] / df["rolling_vol_63"]
    df["drawdown_63"] = df["price"] / df["price"].rolling(63).max() - 1
    df["target_next_return"] = df["return"].shift(-1)

    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    return df


data = simulate_warning_case_market(config)

data.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["date"], data["price"])
plt.title("Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(data["date"], data["return"])
plt.title("Synthetic Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(data["date"], data["regime"], drawstyle="steps-post")
plt.title("Synthetic Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Basic return diagnostics

Return forecasting is hard because:

- mean return is tiny;
- volatility dominates;
- target autocorrelation is weak;
- distribution is heavy-tailed.

These diagnostics help frame expectations.

In [ ]:
def autocorrelation(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.nansum(x**2)

    rows = []
    for lag in range(1, max_lag + 1):
        rows.append({
            "lag": lag,
            "autocorrelation": float(np.nansum(x[lag:] * x[:-lag]) / denom)
        })

    return pd.DataFrame(rows)


acf_ret = autocorrelation(data["return"], 40).assign(series="return")
acf_abs = autocorrelation(data["abs_return"], 40).assign(series="absolute_return")
acf_sq = autocorrelation(data["squared_return"], 40).assign(series="squared_return")
acf_table = pd.concat([acf_ret, acf_abs, acf_sq], ignore_index=True)

summary_stats = pd.Series({
    "daily_mean_return": data["return"].mean(),
    "daily_volatility": data["return"].std(),
    "annualised_mean_return": data["return"].mean() * config.annualisation,
    "annualised_volatility": data["return"].std() * np.sqrt(config.annualisation),
    "skew": data["return"].skew(),
    "excess_kurtosis": data["return"].kurtosis(),
    "target_autocorr_lag1": data["target_next_return"].autocorr(lag=1)
})

summary_stats

In [ ]:
plt.figure(figsize=(10, 5))
for name, group in acf_table.groupby("series"):
    plt.plot(group["lag"], group["autocorrelation"], marker="o", label=name)
plt.axhline(0, linestyle="--")
plt.title("Autocorrelation Diagnostics")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(data["return"], bins=100, density=True)
plt.title("Return Distribution")
plt.xlabel("Return")
plt.ylabel("Density")
plt.show()

## 7. Chronological split

The honest split is chronological:

1. train;
2. validation;
3. test.

No random train/test split is used for final evaluation.

The random split is used only as a warning demonstration.

In [ ]:
n = len(data)
train_end = int(config.train_fraction * n)
validation_end = int((config.train_fraction + config.validation_fraction) * n)

train_df = data.iloc[:train_end].copy()
validation_df = data.iloc[train_end:validation_end].copy()
test_df = data.iloc[validation_end:].copy()

split_metadata = {
    "n_total": int(n),
    "n_train": int(len(train_df)),
    "n_validation": int(len(validation_df)),
    "n_test": int(len(test_df)),
    "train_start": str(train_df["date"].iloc[0]),
    "train_end": str(train_df["date"].iloc[-1]),
    "validation_start": str(validation_df["date"].iloc[0]),
    "validation_end": str(validation_df["date"].iloc[-1]),
    "test_start": str(test_df["date"].iloc[0]),
    "test_end": str(test_df["date"].iloc[-1])
}

pd.Series(split_metadata)

## 8. Feature set and train-only scaling

Features are all backward-looking.

Scaling parameters are fit on training data only:

$$
z=\frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

Validation and test use the same train scaler.

In [ ]:
feature_cols = [
    "return",
    "abs_return",
    "momentum_5",
    "momentum_21",
    "momentum_63",
    "rolling_mean_5",
    "rolling_mean_21",
    "rolling_vol_21",
    "rolling_vol_63",
    "vol_ratio_21_63",
    "drawdown_63"
]

target_col = "target_next_return"

def fit_scaler(df, cols):
    mean = df[cols].mean()
    std = df[cols].std(ddof=1).replace(0, 1.0)
    return mean, std


feature_mean, feature_std = fit_scaler(train_df, feature_cols)
target_mean = train_df[target_col].mean()
target_std = train_df[target_col].std(ddof=1)

def apply_scaling(df, feature_cols, feature_mean, feature_std, target_mean, target_std):
    out = df.copy()

    for col in feature_cols:
        out[f"{col}_scaled"] = (out[col] - feature_mean[col]) / feature_std[col]

    out["target_scaled"] = (out[target_col] - target_mean) / target_std

    return out


scaled_data = apply_scaling(data, feature_cols, feature_mean, feature_std, target_mean, target_std)
scaled_feature_cols = [f"{c}_scaled" for c in feature_cols]

scaled_data[["date"] + scaled_feature_cols[:5] + ["target_scaled"]].head()

## 9. Sequence-window dataset

Each sample contains a trailing window of length $L$:

$$
X_t \in \mathbb{R}^{L\times d}
$$

The label is the next return:

$$
y_t=r_{t+1}
$$

In [ ]:
def make_sequence_dataset(df, feature_cols, target_col, sequence_length):
    values = df[feature_cols].to_numpy(dtype=float)
    targets = df[target_col].to_numpy(dtype=float)
    dates = pd.to_datetime(df["date"]).to_numpy()

    X = []
    y = []
    ts = []

    for t in range(sequence_length - 1, len(df)):
        X.append(values[t - sequence_length + 1:t + 1])
        y.append(targets[t])
        ts.append(dates[t])

    return np.asarray(X), np.asarray(y), np.asarray(ts)


X_all, y_all, ts_all = make_sequence_dataset(
    scaled_data,
    scaled_feature_cols,
    "target_scaled",
    config.sequence_length
)

sequence_meta = pd.DataFrame({"date": pd.to_datetime(ts_all)})

train_end_date = train_df["date"].iloc[-1]
validation_end_date = validation_df["date"].iloc[-1]

train_mask = sequence_meta["date"] <= train_end_date
validation_mask = (sequence_meta["date"] > train_end_date) & (sequence_meta["date"] <= validation_end_date)
test_mask = sequence_meta["date"] > validation_end_date

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_validation, y_validation = X_all[validation_mask], y_all[validation_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
ts_test = sequence_meta.loc[test_mask, "date"].to_numpy()

pd.Series({
    "X_train_shape": str(X_train.shape),
    "X_validation_shape": str(X_validation.shape),
    "X_test_shape": str(X_test.shape)
})

## 10. Baseline forecasts

Baselines:

1. zero forecast;
2. train mean forecast;
3. last return forecast;
4. rolling mean forecast;
5. linear ridge on last-step features.

A deep model must beat these before it earns attention.

In [ ]:
def inverse_target_scale(y_scaled):
    return y_scaled * target_std + target_mean


test_target = inverse_target_scale(y_test)

baseline_forecasts = pd.DataFrame({
    "date": pd.to_datetime(ts_test),
    "actual_next_return": test_target
})

# Align with source rows by date.
test_rows_by_date = scaled_data.set_index("date").loc[pd.to_datetime(ts_test)].reset_index()

baseline_forecasts["forecast_zero"] = 0.0
baseline_forecasts["forecast_train_mean"] = train_df[target_col].mean()
baseline_forecasts["forecast_last_return"] = test_rows_by_date["return"].to_numpy()
baseline_forecasts["forecast_rolling_mean_20"] = test_rows_by_date["return"].rolling(20).mean().fillna(train_df[target_col].mean()).to_numpy()

def fit_ridge(X, y, ridge=1e-3):
    X_design = np.column_stack([np.ones(len(X)), X])
    penalty = ridge * np.eye(X_design.shape[1])
    penalty[0, 0] = 0.0
    beta = np.linalg.solve(X_design.T @ X_design + penalty, X_design.T @ y)
    return beta

def predict_ridge(beta, X):
    X_design = np.column_stack([np.ones(len(X)), X])
    return X_design @ beta

# Linear model on last observation in sequence.
X_train_last = X_train[:, -1, :]
X_test_last = X_test[:, -1, :]
linear_beta = fit_ridge(X_train_last, y_train, ridge=1e-3)
linear_pred_scaled = predict_ridge(linear_beta, X_test_last)
baseline_forecasts["forecast_linear_last_step"] = inverse_target_scale(linear_pred_scaled)

baseline_forecasts.head()

## 11. Forecast metrics

Metrics:

- MSE;
- MAE;
- forecast/actual correlation;
- directional accuracy;
- toy sign-strategy IR.

Toy sign strategy:

$$
R_t = sign(\hat r_{t+1})r_{t+1}
$$

This is diagnostic only.

In [ ]:
def forecast_metrics(df, forecast_col, target_col="actual_next_return"):
    work = df.dropna(subset=[forecast_col, target_col]).copy()
    y = work[target_col].to_numpy()
    pred = work[forecast_col].to_numpy()
    err = pred - y
    strat = np.sign(pred) * y

    return {
        "forecast": forecast_col,
        "n": int(len(work)),
        "mse": float(np.mean(err**2)),
        "mae": float(np.mean(np.abs(err))),
        "corr": float(np.corrcoef(pred, y)[0, 1]) if np.std(pred) > 0 and np.std(y) > 0 else np.nan,
        "directional_accuracy": float(np.mean(np.sign(pred) == np.sign(y))),
        "mean_forecast": float(np.mean(pred)),
        "std_forecast": float(np.std(pred, ddof=1)),
        "toy_sign_ir": float(np.mean(strat) / np.std(strat, ddof=1) * np.sqrt(config.annualisation)) if np.std(strat, ddof=1) > 0 else np.nan
    }


baseline_cols = [
    "forecast_zero",
    "forecast_train_mean",
    "forecast_last_return",
    "forecast_rolling_mean_20",
    "forecast_linear_last_step"
]

baseline_metrics = pd.DataFrame([
    forecast_metrics(baseline_forecasts, col)
    for col in baseline_cols
]).sort_values("mse")

baseline_metrics

## 12. PyTorch datasets and loaders

If PyTorch is available, we train both models.

If not, the notebook still produces baselines and writes a clear model card.

In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(config.seed)

    def to_tensor(x):
        return torch.tensor(x, dtype=torch.float32)

    X_train_t = to_tensor(X_train)
    y_train_t = to_tensor(y_train).view(-1, 1)

    X_validation_t = to_tensor(X_validation)
    y_validation_t = to_tensor(y_validation).view(-1, 1)

    X_test_t = to_tensor(X_test)
    y_test_t = to_tensor(y_test).view(-1, 1)

    class ArrayDataset(torch.utils.data.Dataset):
        def __init__(self, X, y):
            self.X = X
            self.y = y

        def __len__(self):
            return self.X.shape[0]

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    train_loader = torch.utils.data.DataLoader(
        ArrayDataset(X_train_t, y_train_t),
        batch_size=config.batch_size,
        shuffle=True
    )
else:
    X_train_t = y_train_t = X_validation_t = y_validation_t = X_test_t = y_test_t = None
    train_loader = None
    print("PyTorch unavailable: deep models will be skipped.")

## 13. LSTM model

The LSTM receives:

$$
X \in \mathbb R^{batch \times L \times d}
$$

It outputs one scaled next-return forecast.

In [ ]:
if TORCH_AVAILABLE:
    class LSTMForecaster(nn.Module):
        def __init__(self, input_dim, hidden_dim=32, dropout=0.15):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=1,
                batch_first=True
            )
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            )

        def forward(self, x):
            out, _ = self.lstm(x)
            last = out[:, -1, :]
            return self.head(last)
else:
    print("Skipping LSTM model definition because PyTorch unavailable.")

## 14. Transformer encoder model

The Transformer uses:

1. input projection;
2. positional encoding;
3. Transformer encoder;
4. final-token readout.

Important warning:

> A Transformer has more capacity and weaker inductive bias for small financial datasets. This can help or hurt.

In [ ]:
if TORCH_AVAILABLE:
    class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=512):
            super().__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer("pe", pe.unsqueeze(0))

        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]

    class TransformerForecaster(nn.Module):
        def __init__(self, input_dim, d_model=32, nhead=4, num_layers=2, dropout=0.15):
            super().__init__()
            self.input_proj = nn.Linear(input_dim, d_model)
            self.positional = PositionalEncoding(d_model)

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                batch_first=True,
                activation="gelu"
            )

            self.encoder = nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )

            self.head = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model, 1)
            )

        def forward(self, x):
            z = self.input_proj(x)
            z = self.positional(z)
            enc = self.encoder(z)
            last = enc[:, -1, :]
            return self.head(last)
else:
    print("Skipping Transformer model definition because PyTorch unavailable.")

## 15. Parameter count

Large parameter counts can be dangerous when the signal is weak.

We compare model size before training.

In [ ]:
def count_parameters(model):
    if model is None:
        return np.nan
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


input_dim = len(scaled_feature_cols)

if TORCH_AVAILABLE:
    lstm_model = LSTMForecaster(
        input_dim=input_dim,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout
    )

    transformer_model = TransformerForecaster(
        input_dim=input_dim,
        d_model=config.transformer_dim,
        nhead=config.transformer_heads,
        num_layers=config.transformer_layers,
        dropout=config.dropout
    )
else:
    lstm_model = None
    transformer_model = None

parameter_report = pd.DataFrame([
    {"model": "lstm", "parameters": count_parameters(lstm_model)},
    {"model": "transformer", "parameters": count_parameters(transformer_model)}
])

parameter_report

## 16. Training utility

We train with MSE on scaled returns.

Early model selection is based on validation loss.

The training loop is intentionally compact.

In [ ]:
if TORCH_AVAILABLE:
    def train_torch_model(model, train_loader, X_valid, y_valid, config, model_name):
        optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
        loss_fn = nn.MSELoss()

        rows = []
        best_state = None
        best_valid_loss = np.inf

        for epoch in range(config.epochs):
            model.train()
            batch_losses = []

            for xb, yb in train_loader:
                pred = model(xb)
                loss = loss_fn(pred, yb)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                batch_losses.append(float(loss.detach()))

            model.eval()
            with torch.no_grad():
                valid_pred = model(X_valid)
                valid_loss = loss_fn(valid_pred, y_valid)

            row = {
                "model": model_name,
                "epoch": epoch + 1,
                "train_loss": float(np.mean(batch_losses)),
                "valid_loss": float(valid_loss.detach())
            }
            rows.append(row)

            if row["valid_loss"] < best_valid_loss:
                best_valid_loss = row["valid_loss"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if best_state is not None:
            model.load_state_dict(best_state)

        return model, pd.DataFrame(rows)
else:
    print("Skipping training utility because PyTorch unavailable.")

## 17. Train LSTM and Transformer

This may take a little time on CPU.

The notebook keeps the models small to remain runnable.

In [ ]:
training_history = []

if TORCH_AVAILABLE:
    lstm_model, lstm_history = train_torch_model(
        lstm_model,
        train_loader,
        X_validation_t,
        y_validation_t,
        config,
        "lstm"
    )

    transformer_model, transformer_history = train_torch_model(
        transformer_model,
        train_loader,
        X_validation_t,
        y_validation_t,
        config,
        "transformer"
    )

    training_history = pd.concat([lstm_history, transformer_history], ignore_index=True)
else:
    training_history = pd.DataFrame([{
        "model": "none",
        "epoch": 0,
        "train_loss": np.nan,
        "valid_loss": np.nan,
        "note": "PyTorch unavailable"
    }])

training_history.tail()

In [ ]:
if TORCH_AVAILABLE:
    plt.figure(figsize=(10, 6))
    for model_name, g in training_history.groupby("model"):
        plt.plot(g["epoch"], g["valid_loss"], label=f"{model_name} validation")
    plt.title("Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss on scaled returns")
    plt.legend()
    plt.show()
else:
    print("No PyTorch training curves available.")

## 18. Test predictions

We generate test predictions and inverse-transform them back into returns.

In [ ]:
def predict_torch_model(model, X):
    if not TORCH_AVAILABLE or model is None:
        return np.full(X.shape[0], np.nan)

    model.eval()
    with torch.no_grad():
        pred = model(X).detach().cpu().numpy().reshape(-1)

    return pred


if TORCH_AVAILABLE:
    lstm_pred_scaled = predict_torch_model(lstm_model, X_test_t)
    transformer_pred_scaled = predict_torch_model(transformer_model, X_test_t)
else:
    lstm_pred_scaled = np.full(len(y_test), np.nan)
    transformer_pred_scaled = np.full(len(y_test), np.nan)

baseline_forecasts["forecast_lstm"] = inverse_target_scale(lstm_pred_scaled)
baseline_forecasts["forecast_transformer"] = inverse_target_scale(transformer_pred_scaled)

all_forecast_cols = baseline_cols + ["forecast_lstm", "forecast_transformer"]

test_metrics = pd.DataFrame([
    forecast_metrics(baseline_forecasts, col)
    for col in all_forecast_cols
]).sort_values("mse")

test_metrics

In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(test_metrics["forecast"], test_metrics["corr"])
plt.axvline(0, linestyle="--")
plt.title("Chronological Test Forecast Correlation")
plt.xlabel("Correlation")
plt.ylabel("Forecast")
plt.show()

plt.figure(figsize=(12, 6))
plt.barh(test_metrics["forecast"], test_metrics["mse"])
plt.title("Chronological Test MSE")
plt.xlabel("MSE")
plt.ylabel("Forecast")
plt.show()

## 19. Forecast visualisation

Return forecasts should be small.

If a model produces large confident forecasts, that is often overfitting.

We plot a sample of test forecasts.

In [ ]:
plot_sample = baseline_forecasts.iloc[:300]

plt.figure(figsize=(12, 6))
plt.plot(plot_sample["date"], plot_sample["actual_next_return"], label="Actual next return", alpha=0.65)
plt.plot(plot_sample["date"], plot_sample["forecast_lstm"], label="LSTM", alpha=0.85)
plt.plot(plot_sample["date"], plot_sample["forecast_transformer"], label="Transformer", alpha=0.85)
plt.axhline(0, linestyle="--")
plt.title("Chronological Test Forecast Sample")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.show()

## 20. Random split warning case

Now we intentionally create a random split over sequence samples.

This is a warning case, not a valid final benchmark.

Random splits can make nearby time samples appear in both train and validation/test.

For financial series, that can exaggerate performance.

In [ ]:
def make_random_sequence_split(X, y, train_frac=0.60, valid_frac=0.20, seed=123):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(X))
    rng.shuffle(idx)

    n_train = int(train_frac * len(idx))
    n_valid = int(valid_frac * len(idx))

    train_idx = idx[:n_train]
    valid_idx = idx[n_train:n_train + n_valid]
    test_idx = idx[n_train + n_valid:]

    return train_idx, valid_idx, test_idx


random_train_idx, random_valid_idx, random_test_idx = make_random_sequence_split(
    X_all,
    y_all,
    train_frac=config.train_fraction,
    valid_frac=config.validation_fraction,
    seed=config.seed + 999
)

random_split_report = pd.Series({
    "n_random_train": len(random_train_idx),
    "n_random_valid": len(random_valid_idx),
    "n_random_test": len(random_test_idx),
    "warning": "Random sequence split is shown only as a leakage/optimism warning."
})

random_split_report

## 21. Small random-split benchmark

To keep runtime manageable, we train a small Transformer on the random split and compare its random-test performance.

This is labelled clearly as invalid for final model assessment.

In [ ]:
random_warning_metrics = pd.DataFrame()

if TORCH_AVAILABLE:
    X_rand_train = to_tensor(X_all[random_train_idx])
    y_rand_train = to_tensor(y_all[random_train_idx]).view(-1, 1)

    X_rand_valid = to_tensor(X_all[random_valid_idx])
    y_rand_valid = to_tensor(y_all[random_valid_idx]).view(-1, 1)

    X_rand_test = to_tensor(X_all[random_test_idx])
    y_rand_test = y_all[random_test_idx]

    random_loader = torch.utils.data.DataLoader(
        ArrayDataset(X_rand_train, y_rand_train),
        batch_size=config.batch_size,
        shuffle=True
    )

    random_transformer = TransformerForecaster(
        input_dim=input_dim,
        d_model=config.transformer_dim,
        nhead=config.transformer_heads,
        num_layers=config.transformer_layers,
        dropout=config.dropout
    )

    random_transformer, random_transformer_history = train_torch_model(
        random_transformer,
        random_loader,
        X_rand_valid,
        y_rand_valid,
        config,
        "random_split_transformer"
    )

    random_pred_scaled = predict_torch_model(random_transformer, X_rand_test)
    random_pred = inverse_target_scale(random_pred_scaled)
    random_actual = inverse_target_scale(y_rand_test)

    random_warning_df = pd.DataFrame({
        "actual_next_return": random_actual,
        "forecast_random_split_transformer": random_pred
    })

    random_warning_metrics = pd.DataFrame([
        forecast_metrics(
            random_warning_df,
            "forecast_random_split_transformer",
            target_col="actual_next_return"
        )
    ])

else:
    random_transformer_history = pd.DataFrame([{"note": "PyTorch unavailable"}])
    random_warning_metrics = pd.DataFrame([{"note": "PyTorch unavailable"}])

random_warning_metrics

## 22. Chronological versus random split comparison

If random-split performance looks much better, that is not automatically evidence of a better model.

It may indicate split contamination and temporal similarity.

In [ ]:
chron_transformer_metrics = test_metrics[test_metrics["forecast"] == "forecast_transformer"].copy()
chron_transformer_metrics["benchmark_type"] = "chronological_test"

if "forecast" in random_warning_metrics.columns:
    random_compare = random_warning_metrics.copy()
    random_compare["benchmark_type"] = "random_split_warning_case"
    random_compare["forecast"] = "forecast_transformer"
    split_warning_comparison = pd.concat([
        chron_transformer_metrics,
        random_compare[chron_transformer_metrics.columns]
    ], ignore_index=True)
else:
    split_warning_comparison = pd.DataFrame([{"note": "PyTorch unavailable"}])

split_warning_comparison

## 23. Placebo target check

We train on shuffled labels.

If the model still performs well, something is wrong.

A valid sequence model should not generalise meaningfully from randomly permuted targets.

In [ ]:
placebo_metrics = pd.DataFrame()

if TORCH_AVAILABLE:
    rng = np.random.default_rng(config.seed + 2024)
    y_train_placebo = y_train.copy()
    rng.shuffle(y_train_placebo)

    y_train_placebo_t = to_tensor(y_train_placebo).view(-1, 1)

    placebo_loader = torch.utils.data.DataLoader(
        ArrayDataset(X_train_t, y_train_placebo_t),
        batch_size=config.batch_size,
        shuffle=True
    )

    placebo_transformer = TransformerForecaster(
        input_dim=input_dim,
        d_model=config.transformer_dim,
        nhead=config.transformer_heads,
        num_layers=config.transformer_layers,
        dropout=config.dropout
    )

    placebo_transformer, placebo_history = train_torch_model(
        placebo_transformer,
        placebo_loader,
        X_validation_t,
        y_validation_t,
        config,
        "placebo_transformer"
    )

    placebo_pred_scaled = predict_torch_model(placebo_transformer, X_test_t)
    baseline_forecasts["forecast_placebo_transformer"] = inverse_target_scale(placebo_pred_scaled)

    placebo_metrics = pd.DataFrame([
        forecast_metrics(baseline_forecasts, "forecast_transformer"),
        forecast_metrics(baseline_forecasts, "forecast_placebo_transformer"),
        forecast_metrics(baseline_forecasts, "forecast_zero")
    ])

else:
    placebo_history = pd.DataFrame([{"note": "PyTorch unavailable"}])
    placebo_metrics = pd.DataFrame([{"note": "PyTorch unavailable"}])

placebo_metrics

## 24. Regime-dependent errors

A model can perform okay on average and fail in high-volatility regimes.

We bucket test observations by realised absolute return and compare errors.

In [ ]:
regime_df = baseline_forecasts.copy()
regime_df["abs_actual_return"] = regime_df["actual_next_return"].abs()
regime_df["realized_vol_bucket"] = pd.qcut(
    regime_df["abs_actual_return"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"],
    duplicates="drop"
)

regime_rows = []

for bucket, g in regime_df.groupby("realized_vol_bucket", observed=True):
    for col in ["forecast_zero", "forecast_linear_last_step", "forecast_lstm", "forecast_transformer"]:
        if col not in g.columns:
            continue
        y = g["actual_next_return"].to_numpy()
        pred = g[col].to_numpy()
        mask = np.isfinite(pred)
        y = y[mask]
        pred = pred[mask]
        err = pred - y

        regime_rows.append({
            "bucket": str(bucket),
            "forecast": col,
            "n": len(y),
            "mse": float(np.mean(err**2)) if len(y) else np.nan,
            "mae": float(np.mean(np.abs(err))) if len(y) else np.nan,
            "directional_accuracy": float(np.mean(np.sign(pred) == np.sign(y))) if len(y) else np.nan
        })

regime_error = pd.DataFrame(regime_rows)

regime_error

In [ ]:
plt.figure(figsize=(10, 6))
for forecast, g in regime_error.groupby("forecast"):
    plt.plot(g["bucket"], g["mse"], marker="o", label=forecast)
plt.title("Forecast MSE by Realised Return Bucket")
plt.xlabel("Realised absolute return bucket")
plt.ylabel("MSE")
plt.legend()
plt.show()

## 25. Toy strategy diagnostic

We convert forecasts into simple weights:

$$
w_t = clip\Big(\frac{\hat r_{t+1}}{2\sigma_{\hat r}}, -1, 1\Big)
$$

Net return proxy:

$$
R_t = w_t r_{t+1} - c|\Delta w_t|
$$

This is not a real backtest.

It is a sanity check for whether forecast signs and magnitudes have economic shape after turnover costs.

In [ ]:
def toy_strategy(df, forecast_col, cost_per_turnover=0.0005):
    out = df[["date", "actual_next_return", forecast_col]].dropna().copy()
    pred = out[forecast_col].to_numpy()

    scale = np.std(pred, ddof=1)
    scale = scale if scale > 1e-12 else 1.0

    weight = np.clip(pred / (2 * scale), -1.0, 1.0)
    turnover = np.abs(np.diff(np.r_[0.0, weight]))

    gross = weight * out["actual_next_return"].to_numpy()
    cost = cost_per_turnover * turnover
    net = gross - cost

    out["weight"] = weight
    out["turnover"] = turnover
    out["gross_return_proxy"] = gross
    out["cost"] = cost
    out["net_return_proxy"] = net
    out["cum_net_return_proxy"] = np.cumsum(net)

    summary = {
        "forecast": forecast_col,
        "cost_per_turnover": cost_per_turnover,
        "mean_abs_weight": float(np.mean(np.abs(weight))),
        "mean_turnover": float(np.mean(turnover)),
        "total_gross_return_proxy": float(np.sum(gross)),
        "total_net_return_proxy": float(np.sum(net)),
        "mean_net_return_proxy": float(np.mean(net)),
        "vol_net_return_proxy": float(np.std(net, ddof=1)),
        "ir_proxy": float(np.mean(net) / np.std(net, ddof=1) * np.sqrt(config.annualisation)) if np.std(net, ddof=1) > 0 else np.nan
    }

    return out, summary


strategy_cols = [
    "forecast_zero",
    "forecast_linear_last_step",
    "forecast_lstm",
    "forecast_transformer"
]

strategy_frames = []
strategy_summaries = []

for col in strategy_cols:
    strat, summary = toy_strategy(baseline_forecasts, col, config.cost_per_turnover)
    strat["forecast"] = col
    strategy_frames.append(strat)
    strategy_summaries.append(summary)

strategy_df = pd.concat(strategy_frames, ignore_index=True)
strategy_summary = pd.DataFrame(strategy_summaries).sort_values("ir_proxy", ascending=False)

strategy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for forecast, g in strategy_df.groupby("forecast"):
    plt.plot(g["date"], g["cum_net_return_proxy"], label=forecast)
plt.axhline(0, linestyle="--")
plt.title("Toy Strategy Diagnostic")
plt.xlabel("Date")
plt.ylabel("Cumulative net return proxy")
plt.legend()
plt.show()

## 26. Cost sensitivity

A model with higher turnover needs stronger predictive power.

We vary cost assumptions and compare LSTM and Transformer.

In [ ]:
cost_grid = [0.0, 0.0001, 0.00025, 0.0005, 0.0010, 0.0020]
cost_rows = []

for cost in cost_grid:
    for col in ["forecast_linear_last_step", "forecast_lstm", "forecast_transformer"]:
        _, summary = toy_strategy(baseline_forecasts, col, cost)
        cost_rows.append(summary)

cost_sensitivity = pd.DataFrame(cost_rows)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for forecast, g in cost_sensitivity.groupby("forecast"):
    plt.plot(g["cost_per_turnover"], g["ir_proxy"], marker="o", label=forecast)
plt.axhline(0, linestyle="--")
plt.title("Toy Strategy Cost Sensitivity")
plt.xlabel("Cost per turnover")
plt.ylabel("IR proxy")
plt.legend()
plt.show()

## 27. Model card and warning labels

A responsible comparison should report:

| Item | Status |
|---|---|
| Forecast target | next return |
| Final split | chronological |
| Random split | warning case only |
| Baselines | included |
| Placebo test | included |
| Cost diagnostic | included |
| Claim of tradable alpha | not made |
| Claim Transformer is superior | not made |

This notebook is designed to prevent architecture hype from replacing validation.

In [ ]:
model_card = pd.Series({
    "notebook": "03_17_transformer_vs_lstm_for_returns_warning_case",
    "torch_available": TORCH_AVAILABLE,
    "target": "next_period_return",
    "final_split": "chronological",
    "random_split_used_as": "warning_case_only",
    "baselines_included": True,
    "placebo_check_included": True,
    "regime_error_included": True,
    "cost_sensitivity_included": True,
    "claim_transformer_superiority": "not made",
    "claim_tradable_alpha": "not made"
})

model_card

## 28. Practical checklist for Transformer/LSTM return forecasting

Before believing a deep return-forecasting result, check:

1. **Chronological split**  
   Was final test evaluation time-ordered?

2. **No random split claims**  
   Is random split used only as a warning or ablation?

3. **Train-only scaling**  
   Were scalers fit only on train?

4. **Baselines**  
   Did the model beat zero, mean, and simple linear forecasts?

5. **Placebo target**  
   Does performance disappear under shuffled labels?

6. **Parameter count**  
   Is the model too large for the dataset?

7. **Regime errors**  
   Does it fail in high-volatility periods?

8. **Cost sensitivity**  
   Does any apparent edge survive turnover costs?

9. **Forecast magnitude**  
   Are predictions unrealistically confident?

10. **Economic rationale**  
   Is there a reason the sequence structure should predict returns?

11. **Reproducibility**  
   Are seeds, splits, and configs saved?

12. **No hype**  
   Do not say attention predicts markets unless the evidence survives hard validation.

## 29. Saving outputs

The notebook saves:

1. synthetic market data;
2. autocorrelation diagnostics;
3. split metadata;
4. parameter report;
5. training history;
6. chronological test forecasts;
7. test metrics;
8. random split warning metrics;
9. placebo metrics;
10. regime error report;
11. toy strategy results;
12. cost sensitivity;
13. model card;
14. manifest.

In [ ]:
output_dir = Path("data/processed/transformer_vs_lstm_for_returns_warning_case_v1")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = output_dir / "synthetic_market_data.csv"
acf_path = output_dir / "autocorrelation_diagnostics.csv"
summary_stats_path = output_dir / "summary_stats.csv"
split_metadata_path = output_dir / "split_metadata.json"
parameter_report_path = output_dir / "parameter_report.csv"
training_history_path = output_dir / "training_history.csv"
test_forecasts_path = output_dir / "chronological_test_forecasts.csv"
test_metrics_path = output_dir / "chronological_test_metrics.csv"
random_warning_metrics_path = output_dir / "random_split_warning_metrics.csv"
split_warning_comparison_path = output_dir / "split_warning_comparison.csv"
placebo_metrics_path = output_dir / "placebo_metrics.csv"
regime_error_path = output_dir / "regime_error.csv"
strategy_summary_path = output_dir / "toy_strategy_summary.csv"
strategy_df_path = output_dir / "toy_strategy_paths.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
model_card_path = output_dir / "model_card.csv"
manifest_path = output_dir / "manifest.json"

data.to_csv(data_path, index=False)
acf_table.to_csv(acf_path, index=False)
summary_stats.to_frame("value").to_csv(summary_stats_path)
Path(split_metadata_path).write_text(json.dumps(split_metadata, indent=2, default=str))
parameter_report.to_csv(parameter_report_path, index=False)
training_history.to_csv(training_history_path, index=False)
baseline_forecasts.to_csv(test_forecasts_path, index=False)
test_metrics.to_csv(test_metrics_path, index=False)
random_warning_metrics.to_csv(random_warning_metrics_path, index=False)
split_warning_comparison.to_csv(split_warning_comparison_path, index=False)
placebo_metrics.to_csv(placebo_metrics_path, index=False)
regime_error.to_csv(regime_error_path, index=False)
strategy_summary.to_csv(strategy_summary_path, index=False)
strategy_df.to_csv(strategy_df_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
model_card.to_frame("value").to_csv(model_card_path)

manifest = {
    "dataset_name": "transformer_vs_lstm_for_returns_warning_case_outputs",
    "schema_version": "transformer_vs_lstm_for_returns_warning_case_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "torch_available": TORCH_AVAILABLE,
    "feature_cols": feature_cols,
    "scaled_feature_cols": scaled_feature_cols,
    "split_metadata": split_metadata,
    "parameter_report": parameter_report.to_dict(orient="records"),
    "chronological_test_metrics": test_metrics.to_dict(orient="records"),
    "model_card": model_card.to_dict(),
    "notes": [
        "Synthetic market contains weak, unstable predictability and volatility regimes.",
        "Final evaluation uses chronological train/validation/test split.",
        "Random split is included only as a warning case.",
        "Baselines include zero, train mean, last return, rolling mean, and linear last-step ridge.",
        "LSTM and Transformer models are trained only if PyTorch is available.",
        "Placebo target check, regime error diagnostics, and cost sensitivity are included.",
        "Notebook explicitly avoids claiming that Transformer or LSTM predicts markets reliably."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

data_path, test_metrics_path, random_warning_metrics_path, model_card_path, manifest_path

## 30. Limitations

### 30.1 Synthetic data

The notebook uses synthetic data.

Real markets require data cleaning, corporate actions, contract rolls, survivorship controls, liquidity filters, and accurate timestamps.

### 30.2 Small model comparison

The LSTM and Transformer are compact.

This is not a state-of-the-art deep learning benchmark.

### 30.3 Return forecasting remains weak

Even when synthetic data contains weak signal, return predictability is fragile.

Real data is usually harder.

### 30.4 No purged overlapping-label CV

The target is next-period return, so overlap is limited.

For multi-day labels, use purged cross-validation from notebook `03_15`.

### 30.5 Random split is intentionally invalid

The random split result is a warning case, not a valid performance estimate.

### 30.6 Toy strategy is simplified

The strategy ignores real execution, slippage, market impact, leverage constraints, borrow, and capacity.

### 30.7 No hyperparameter sweep

A full comparison would require nested, purged, walk-forward tuning.

### 30.8 PyTorch dependency

If PyTorch is unavailable, deep models are skipped and the notebook reports that clearly.

## 31. Research frontier and extensions

Important extensions include:

1. **Purged walk-forward validation**  
   Combine deep sequence models with leakage-safe model selection.

2. **Temporal convolutional networks**  
   Often more stable than LSTMs/Transformers on small sequences.

3. **Probabilistic forecasts**  
   Predict distributions or quantiles rather than point returns.

4. **Multi-asset Transformers**  
   Model cross-asset context, but with careful validation.

5. **Regime-conditioned neural models**  
   Switch behaviour by volatility or HMM regime.

6. **Cost-aware loss functions**  
   Optimise a differentiable trading utility, not MSE.

7. **Representation learning for features**  
   Pretrain on many assets before return forecasting.

8. **Model uncertainty**  
   Ensembles, dropout, or Bayesian approximations.

9. **Attention diagnostics**  
   Interpret whether attention is meaningful or noisy.

10. **Chinese futures sequence models**  
   Include night sessions, L1 microstructure, roll calendars, price limits, and product-specific liquidity.

## 32. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_03_risk_model_factor_exposures**  
   Separate alpha from factor risk.

2. **05_05_walk_forward_model_validation**  
   General live-like validation infrastructure.

3. **05_06_combinatorial_purged_cross_validation**  
   Formal leakage-safe validation.

4. **06_12_gpu_deep_learning_research_stack**  
   Scaling sequence training.

5. **07_01_multi_asset_cta_research_pipeline**  
   Use deep sequence models only as one small component of a full research system.

6. **Model risk capstone**  
   Compare classical, tree, LSTM, Transformer, and ensemble models under strict validation.

## 33. Summary

This notebook implemented a Transformer vs LSTM warning case for return forecasting.

It showed how to:

1. simulate weak and unstable return predictability;
2. create leakage-safe sequence windows;
3. use chronological train/validation/test splits;
4. build simple baselines;
5. train compact LSTM and Transformer models if PyTorch is available;
6. compare parameter counts;
7. evaluate on chronological test data;
8. demonstrate random-split optimism;
9. run placebo-target checks;
10. diagnose regime-dependent errors;
11. convert forecasts into a toy cost-aware strategy;
12. save model cards and reports.

The key computational takeaway:

> A model architecture comparison is meaningless without a validation protocol that matches the time-series problem.

The key financial takeaway:

> In return forecasting, Transformer attention is not evidence of alpha; only out-of-sample, cost-aware, regime-robust performance against baselines matters.

## 34. Further reading

- Vaswani et al., "Attention Is All You Need."
- Hochreiter and Schmidhuber, "Long Short-Term Memory."
- López de Prado, *Advances in Financial Machine Learning.*
- Bailey et al. on backtest overfitting.
- Literature on time-series validation, purged CV, deep learning for finance, and model risk in financial ML.